In [ ]:
import optuna
from optuna.importance import get_param_importances

# CAN ALSO USE optuna-dashboard sqlite:///experiments/results/tuning.db FOR LIVE CHECKING

study = optuna.load_study(
    study_name="als_tuning",
    storage="sqlite:///experiments/results/tuning.db"
)
print(f"Completed trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"Best so far: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:

importances = get_param_importances(study)

print("\nParameter importance:")
for param, score in importances.items():
    print(f"{param:15s}: {score:.4f}")

In [ ]:
import torch
print(torch.__version__)           # Should show 2.x.x+cu124
print(torch.cuda.is_available())   # Should return True
print(torch.cuda.get_device_name(0))  # Should show "NVIDIA GeForce RTX 4080"

In [ ]:
import torch
from data import build_dataset
from models.lightgcn import LightGCNModel

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

# Small smoke test — 2 epochs, tiny batch
ds = build_dataset(split="leave_last_n", k=10, lln_n=2)

model = LightGCNModel(
    n_users=ds.n_users,
    n_items=ds.n_items,
    embedding_dim=32,
    n_layers=3,
    n_epochs=5,
    batch_size=16384,
)

print(f"Model device: {model.device}")
torch.cuda.empty_cache()
model.fit(ds.train_df)
print("Smoke test passed.")

CUDA available: True
Device: NVIDIA GeForce RTX 4080
Model device: cuda
Building seen dict …
Building CSR seen structure …
Building normalised adjacency …
  Epoch   1/5  loss=0.9002
  Epoch   2/5  loss=0.2580
  Epoch   3/5  loss=0.2384
  Epoch   4/5  loss=0.2304
  Epoch   5/5  loss=0.2259
Smoke test passed.


In [ ]:
import torch
from data import build_dataset
from models.bpr import BPRModel

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

ds = build_dataset(split="leave_last_n", k=10, lln_n=2)

bpr_model = BPRModel(
    factors=32,
    iterations=10,
    learning_rate=0.05,
    regularization=0.001,
    use_gpu=False
)
torch.cuda.empty_cache()
bpr_model.fit(ds.train_df, ds.implicit_matrix)
print("Smoke test passed.")

CUDA available: True
Device: NVIDIA GeForce RTX 4080
Using GPU with cupy


ValueError: No CUDA extension has been built, can't train on GPU.

In [4]:
print(ds.implicit_matrix.shape)   # (users, items)
print(ds.implicit_matrix.nnz)     # number of interactions

(200947, 31961)
15427055


In [3]:
import implicit
print(implicit.gpu.HAS_CUDA)  # should be True
print(implicit.__version__)

False
0.7.2


In [ ]:
import torch
print(torch.version.cuda)

12.4


In [3]:
import cupy
print(cupy.cuda.runtime.runtimeGetVersion())  # should return 12040 for 12.4
print(cupy.cuda.Device(0).attributes)         # confirms 4080 is accessible

12090
{'AsyncEngineCount': 1, 'CanFlushRemoteWrites': 0, 'CanMapHostMemory': 1, 'CanUseHostPointerForRegisteredMem': 0, 'ClockRate': 2595000, 'ComputeMode': 0, 'ComputePreemptionSupported': 1, 'ConcurrentKernels': 1, 'ConcurrentManagedAccess': 0, 'CooperativeLaunch': 1, 'CooperativeMultiDeviceLaunch': 0, 'DirectManagedMemAccessFromHost': 0, 'EccEnabled': 0, 'GPUDirectRDMAFlushWritesOptions': 0, 'GPUDirectRDMASupported': 0, 'GPUDirectRDMAWritesOrdering': 0, 'GlobalL1CacheSupported': 1, 'GlobalMemoryBusWidth': 256, 'GpuOverlap': 1, 'HostNativeAtomicSupported': 0, 'HostRegisterReadOnlySupported': 1, 'HostRegisterSupported': 1, 'Integrated': 0, 'IsMultiGpuBoard': 0, 'KernelExecTimeout': 1, 'L2CacheSize': 67108864, 'LocalL1CacheSupported': 1, 'ManagedMemory': 1, 'MaxBlockDimX': 1024, 'MaxBlockDimY': 1024, 'MaxBlockDimZ': 64, 'MaxBlocksPerMultiprocessor': 24, 'MaxGridDimX': 2147483647, 'MaxGridDimY': 65535, 'MaxGridDimZ': 65535, 'MaxPitch': 2147483647, 'MaxRegistersPerBlock': 65536, 'MaxRegi

In [ ]:
import time
import cupy

# bpr_model = BPRModel(factors=32, iterations=10, learning_rate=0.05, regularization=0.001, use_gpu=True)

# start = time.perf_counter()
# bpr_model.fit(ds.train_df, ds.implicit_matrix)
# gpu_time = time.perf_counter() - start

bpr_model_cpu = BPRModel(factors=32, iterations=10, learning_rate=0.05, regularization=0.001, use_gpu=False)
start = time.perf_counter()
bpr_model_cpu.fit(ds.train_df, ds.implicit_matrix)
cpu_time = time.perf_counter() - start

# print(f"GPU: {gpu_time:.1f}s  CPU: {cpu_time:.1f}s  speedup: {cpu_time/gpu_time:.2f}x")

ValueError: No CUDA extension has been built, can't train on GPU.